In [1]:
import pandas as pd
import os
import shutil
import pyarrow.parquet as pq

# =====================================================================
# 1. CONSTANTES E CONFIGURAÇÕES DE CAMINHO
# =====================================================================

ARQ_BASE_PATH = r'C:\Users\Pedro\OneDrive\Villam\POWERBI\mds_Notas\Relatorio Notas\Base_Limpa.parquet'
INCREMENTAIS_DIR = r'C:\Users\Pedro\OneDrive\Villam\POWERBI\mds_Notas\Relatorio Notas\incrementais'
SAIDA_FINAL_PATH = r'C:\Users\Pedro\OneDrive\Villam\POWERBI\mds_Notas\Relatorio Notas\Base_Limpa.parquet' # Sobrescreve a base original
JA_PROCESSADOS_DIR = os.path.join(INCREMENTAIS_DIR, 'ja_processados')

# Nomes de Arquivos Incrementais
INC_NOTA = 'notas_fiscais_22_02_2026_01_03_2026.csv'
INC_GUIAS = 'notas_fiscais_guias_22_02_2026_01_03_2026.csv'
INC_CANC = 'notas_fiscais_canceladas_22_02_2026_01_03_2026.csv'

# Caminhos Completos
INC_NOTA_PATH = os.path.join(INCREMENTAIS_DIR, INC_NOTA)
INC_GUIAS_PATH = os.path.join(INCREMENTAIS_DIR, INC_GUIAS)
INC_CANC_PATH = os.path.join(INCREMENTAIS_DIR, INC_CANC)

# Chaves de Controle
CHAVE_UNICA = ['Nota', 'CPF_CNPJ_Prestador', 'CPF_CNPJ_Tomador']
CHAVE_CANCELAMENTO = ['Nota', 'CPF_CNPJ_Prestador']
COLUNA_GUIA = 'Numero_Guia'
COLUNA_PRESTADOR = 'CPF_CNPJ_Prestador'
COLUNA_DATA = 'Data_Emissao'

# Listas de Filtro e Tipagem
CNPJS_TESTE = [
    '21004012000164','77580066000122','37249433000195','47621331000102','19411731000158',
    '12216767000131','11898201000174','12221498000100','21877832000160','63838133000151',
    '45738773000108','97624678000187','33918233000127','20720313000121','83165370000106',
    '85809284000114','11670499000160','15997136000195','56373945000103','41592516000150'
]
COLUNAS_FLOAT = [
    'Valor_da_Nota', 
    'Base_de_Calculo', 
    'Valor_Deducoes', 
    'Valor_ISS'
]

In [2]:
# --- Funções Auxiliares ---
def MOVER_ARQ(source_path, category):
    """Move o arquivo incremental processado para a pasta 'ja_processados'."""
    try:
        dest_dir = os.path.join(JA_PROCESSADOS_DIR, category)
        os.makedirs(dest_dir, exist_ok=True)
        filename = os.path.basename(source_path)
        dest_path = os.path.join(dest_dir, filename)
        
        shutil.move(source_path, dest_path)
        print(f"-> Arquivo movido para: {dest_dir}")
    except FileNotFoundError:
        print(f"AVISO: Arquivo incremental não encontrado ou já movido: {source_path}")
    except Exception as e:
        print(f"ERRO ao mover arquivo {source_path}: {e}")

def ler_csv(path):
    """Lê um CSV com configurações robustas para strings."""
    if not os.path.exists(path):
        return None
    return pd.read_csv(path, dtype=str, sep=';', na_filter=False)

# --- Funções de Processamento Incremental ---

def process_notas(df_base, inc_path):
    """Processa o incremental de Adição de Notas (Anti-Join)."""
    notas = ler_csv(inc_path)
    if notas is None or notas.empty:
        print("\n--- Adição de Notas Fiscais: Pulada (Arquivo vazio ou não encontrado). ---")
        return df_base

    print(f"\n--- INICIANDO PROCESSAMENTO: Adição de Notas Fiscais --- (Total: {len(notas)})")
    
    # Anti-Join: Identifica o que é 'left_only' (novo)
    df_verificacao = notas.merge(
        df_base[CHAVE_UNICA], 
        on=CHAVE_UNICA,
        how='left',
        indicator=True
    )
    df_novas_linhas = df_verificacao.query('_merge == "left_only"').drop(columns=['_merge'])
    linhas_ignoradas = len(notas) - len(df_novas_linhas)
    
    if len(df_novas_linhas) > 0:
        df_base = pd.concat([df_base, df_novas_linhas], ignore_index=True)
        print(f"Linhas únicas adicionadas à base: {len(df_novas_linhas)}")
        print(f"Linhas ignoradas por duplicidade: {linhas_ignoradas}")
    else:
        print("Nenhuma linha nova encontrada para adição.")

    MOVER_ARQ(inc_path, 'notas')
    return df_base

def process_guias(df_base, inc_path):
    """Processa o incremental de Guias (Upsert)."""
    guias = ler_csv(inc_path)
    if guias is None or guias.empty:
        print("\n--- Upsert de Guias: Pulado (Arquivo vazio ou não encontrado). ---")
        return df_base

    print(f"\n--- INICIANDO PROCESSAMENTO: Upsert de Guias --- (Total: {len(guias)})")
    
    # PARTE 1: UPDATE (Busca linhas existentes para atualizar a Guia)
    df_para_update = df_base.merge(
        guias[CHAVE_UNICA + [COLUNA_GUIA]],
        on=CHAVE_UNICA,
        how='inner', 
        suffixes=('_base', '_inc')
    )
    
    if not df_para_update.empty:
        # Aplica o novo valor da guia e mantém o schema original
        df_para_update[COLUNA_GUIA] = df_para_update[COLUNA_GUIA + '_inc']
        df_para_update = df_para_update[df_base.columns.tolist()]
        
        # Delete/Insert em memória
        df_base_nao_alterada = df_base.merge(
            df_para_update[CHAVE_UNICA], on=CHAVE_UNICA, how='left', indicator=True
        ).query('_merge == "left_only"').drop(columns=['_merge'])
        df_base = pd.concat([df_base_nao_alterada, df_para_update], ignore_index=True)
        print(f"Total de linhas atualizadas (guia inserida/alterada): {len(df_para_update)}")

    # PARTE 2: INSERT (Adiciona linhas que são totalmente novas)
    df_para_insert = guias.merge(
        df_base[CHAVE_UNICA], on=CHAVE_UNICA, how='left', indicator=True
    ).query('_merge == "left_only"').drop(columns=['_merge'])
    
    if len(df_para_insert) > 0:
        df_base = pd.concat([df_base, df_para_insert], ignore_index=True)
        print(f"Total de linhas NOVAS adicionadas (Insert): {len(df_para_insert)}")
    else:
        print("Nenhuma linha nova encontrada para inserção.")

    MOVER_ARQ(inc_path, 'guias')
    return df_base

def process_canceladas(df_base, inc_path):
    """Processa o incremental de Cancelamento (Atualiza coluna 'Cancelada')."""
    canceladas = ler_csv(inc_path)
    if canceladas is None or canceladas.empty:
        print("\n--- Atualização de Cancelamento: Pulada (Arquivo vazio ou não encontrado). ---")
        return df_base

    print(f"\n--- INICIANDO PROCESSAMENTO: Atualização de Cancelamento ---")
    
    # 1. Identificar as chaves na base que precisam ser canceladas
    chaves_para_marcar = canceladas[CHAVE_CANCELAMENTO].drop_duplicates()
    
    # 2. Criar a máscara para o df_base usando .isin()
    condicao_cancelamento = df_base.set_index(CHAVE_CANCELAMENTO).index.isin(
        chaves_para_marcar.set_index(CHAVE_CANCELAMENTO).index
    )
    
    # 3. Aplicar a atualização diretamente na coluna 'Cancelada'
    linhas_atualizadas = condicao_cancelamento.sum()
    df_base.loc[condicao_cancelamento, 'Cancelada'] = 'SIM'

    print(f"Total de linhas marcadas como 'SIM' na coluna 'Cancelada': {linhas_atualizadas}")

    MOVER_ARQ(inc_path, 'canceladas')
    return df_base

In [3]:
def ajustar_data(df_base):
    """Remove duplicatas por linha inteira e CNPJs de teste."""
    print("\n--- INICIANDO LIMPEZA DE DADOS (Duplicatas e Filtro) ---")
    
    # 1. Remoção de Duplicatas (Linha Inteira)
    linhas_antes_dup = len(df_base)
    df_base = df_base.drop_duplicates(keep='first')
    linhas_removidas_dup = linhas_antes_dup - len(df_base)
    print(f"Duplicatas por linha inteira removidas: {linhas_removidas_dup}")

    # 2. Remoção de CNPJs de Teste
    linhas_antes_cnpjs = len(df_base)
    df_base = df_base[~df_base[COLUNA_PRESTADOR].isin(CNPJS_TESTE)]
    linhas_removidas_cnpjs = linhas_antes_cnpjs - len(df_base)
    print(f"CNPJs de teste removidos: {linhas_removidas_cnpjs}")

    # 3. Remover coluna 'cancelada' antiga (correção de erro anterior)
    df_base.drop(columns=['cancelada'], inplace=True, errors='ignore')
    
    print(f"Novo total de linhas na base após limpeza: {len(df_base)}")
    return df_base

def schema(df_base):
    """Aplica o schema final de tipagem (Float e Data Normalizada)."""
    print("\n--- APLICANDO SCHEMA FINAL E NORMALIZAÇÃO DE DATA ---")
    
    SCHEMA_FINAL = {}

    # 1. Tipagem e Normalização da Data
    df_base[COLUNA_DATA] = pd.to_datetime(
        df_base[COLUNA_DATA], errors='coerce', format='mixed'
    )
    # Zera a hora para 00:00:00
    df_base[COLUNA_DATA] = df_base[COLUNA_DATA].dt.normalize()
    SCHEMA_FINAL[COLUNA_DATA] = 'datetime64[ns]'

     # 1.1. Criar coluna PA (YYYYMM)
    df_base["PA"] = df_base[COLUNA_DATA].dt.strftime("%Y%m").astype("Int64")
    SCHEMA_FINAL["PA"] = "Int64"
    
     # Criar Dia (equivalente ao FORMAT dd)
    df_base["Dia"] = df_base[COLUNA_DATA].dt.day.astype("Int64")
    SCHEMA_FINAL["Dia"] = "Int64"

    # 2. Tipagem de Float e String
    for coluna in df_base.columns:
        if coluna in COLUNAS_FLOAT:
            df_base[coluna] = pd.to_numeric(df_base[coluna], errors='coerce')
            SCHEMA_FINAL[coluna] = 'float64'
        
        elif coluna not in SCHEMA_FINAL:
            SCHEMA_FINAL[coluna] = 'object'

    df_base = df_base.astype(SCHEMA_FINAL, errors='ignore')
    print("Schema aplicado com sucesso.")
    return df_base

def salvar_parquet(df_base, path):
    """Salva a base final no Parquet, forçando o tipo DATE para a coluna de Data."""
    
    # Ajuste Final de Data (Convertendo para o objeto Python 'date')
    # Isso é o que força o PyArrow a inferir o tipo DATE/date32 (data pura).
    df_base[COLUNA_DATA] = df_base[COLUNA_DATA].dt.date.astype('object')
    
    # Salvamento com compressão Snappy
    df_base.to_parquet(
        path, 
        index=False, 
        engine='pyarrow', 
        compression='snappy'
    )
    
    print("\n---------------------------------------------------------")
    print(f"✅ SUCESSO! Base Parquet salva e persistida em: {path}")
    print(f"Total de Linhas Salvas: {len(df_base)}. Compressão: Snappy.")
    print("---------------------------------------------------------")

In [4]:
def main_pipeline():
    """Carrega, processa, limpa e salva a base de dados."""
    
    # Carregamento Inicial
    print("--- INICIANDO PIPELINE DE PROCESSAMENTO INCREMENTAL ---")
    print(f"Carregando base de dados de: {ARQ_BASE_PATH}")
    
    # Inicializa a base e cria a pasta de processados
    df_base = pd.read_parquet(ARQ_BASE_PATH)
    os.makedirs(JA_PROCESSADOS_DIR, exist_ok=True)
    print(f"Total de linhas inicial: {len(df_base)}")

    # 1. Processamento - Adição de Notas
    df_base = process_notas(df_base, INC_NOTA_PATH)

    # 2. Processamento - Upsert de Guias
    df_base = process_guias(df_base, INC_GUIAS_PATH)

    # 3. Processamento - Cancelamento
    df_base = process_canceladas(df_base, INC_CANC_PATH)

    # 4. Limpeza e Filtros
    df_base = ajustar_data(df_base)

    # 5. Aplicação do Schema Final
    df_base = schema(df_base)
    
    # 6. Salvamento Final
    salvar_parquet(df_base, SAIDA_FINAL_PATH)
    
    print("\n--- PIPELINE CONCLUÍDO COM SUCESSO! ---")

# Executa o pipeline
main_pipeline()

--- INICIANDO PIPELINE DE PROCESSAMENTO INCREMENTAL ---
Carregando base de dados de: C:\Users\Pedro\OneDrive\Villam\POWERBI\mds_Notas\Relatorio Notas\Base_Limpa.parquet
Total de linhas inicial: 1534772

--- INICIANDO PROCESSAMENTO: Adição de Notas Fiscais --- (Total: 9271)
Linhas únicas adicionadas à base: 9271
Linhas ignoradas por duplicidade: 0
-> Arquivo movido para: C:\Users\Pedro\OneDrive\Villam\POWERBI\mds_Notas\Relatorio Notas\incrementais\ja_processados\notas

--- INICIANDO PROCESSAMENTO: Upsert de Guias --- (Total: 6116)
Total de linhas atualizadas (guia inserida/alterada): 4454
Total de linhas NOVAS adicionadas (Insert): 1662
-> Arquivo movido para: C:\Users\Pedro\OneDrive\Villam\POWERBI\mds_Notas\Relatorio Notas\incrementais\ja_processados\guias

--- INICIANDO PROCESSAMENTO: Atualização de Cancelamento ---
Total de linhas marcadas como 'SIM' na coluna 'Cancelada': 69
-> Arquivo movido para: C:\Users\Pedro\OneDrive\Villam\POWERBI\mds_Notas\Relatorio Notas\incrementais\ja_proc